In [1]:
STUDENT_NAME = "Drashti Shah"
FULL_SID = "013852046"
SID4, SEED, SLICE, HP_ID, CLS_A, CLS_B = 2046, 2046, 46, 0, 6, 3
assert (SID4, SEED, SLICE, HP_ID, CLS_A, CLS_B) == (2046, 2046, 46, 0, 6, 3)
print(f"Student: {STUDENT_NAME} | Full SID: {FULL_SID}")
print(f"SID4={SID4} SEED={SEED} SLICE={SLICE} HP_ID={HP_ID} CLS_A={CLS_A} CLS_B={CLS_B}")

Student: Drashti Shah | Full SID: 013852046
SID4=2046 SEED=2046 SLICE=46 HP_ID=0 CLS_A=6 CLS_B=3


In [2]:
import os, subprocess, sys, shutil
from datetime import datetime, timezone
from pathlib import Path

os.chdir("/content")  # every subsequent cell in this notebook runs from here
print("Working directory:", os.getcwd())

RUN_LOG = Path("RUN_LOG.txt")

def log(section, message):
    stamp = datetime.now(timezone.utc).isoformat()
    with RUN_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{stamp}] [{section}] {message}\n")

def run_command(command, check=True, save_to=None):
    print("$", " ".join(command))
    completed = subprocess.run(command, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, check=False)
    print(completed.stdout)
    if save_to:
        Path(save_to).write_text(completed.stdout, encoding="utf-8")
    log("CUDA SECTION", f"$ {' '.join(command)}\n{completed.stdout}")
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed ({completed.returncode}): {' '.join(command)}")
    return completed

log("CUDA SECTION START", f"SID4={SID4} SEED={SEED}")

if shutil.which("nvidia-smi") is None or shutil.which("nvcc") is None:
    raise RuntimeError("CUDA tools not found — set Runtime > Change runtime type > GPU (T4).")

run_command(["nvidia-smi"])
run_command(["nvcc", "--version"])

Working directory: /content
$ nvidia-smi
Tue Sep  1 04:48:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------

CompletedProcess(args=['nvcc', '--version'], returncode=0, stdout='nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0\n')

In [3]:
print("Files in", os.getcwd(), ":", os.listdir("."))

if not Path("matrix_mul.cu").exists():
    raise FileNotFoundError("Upload matrix_mul.cu into /content (Colab's file browser root) first.")

run_command(["nvcc", "-O3", "-std=c++17", "-Xcompiler", "-fopenmp",
             "-lineinfo", "matrix_mul.cu", "-o", "matrix_mul"])

Files in /content : ['.config', '.ipynb_checkpoints', 'RUN_LOG.txt', 'matrix_mul.cu', 'matrix_mul', 'sample_data', 'cuda_metrics.csv', 'profiler_output.txt']
$ nvcc -O3 -std=c++17 -Xcompiler -fopenmp -lineinfo matrix_mul.cu -o matrix_mul
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).



CompletedProcess(args=['nvcc', '-O3', '-std=c++17', '-Xcompiler', '-fopenmp', '-lineinfo', 'matrix_mul.cu', '-o', 'matrix_mul'], returncode=0, stdout="nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).\n")

In [4]:
benchmark_run = run_command(["./matrix_mul", "256", "1024", "4096"], save_to="cuda_run_output.txt")

import pandas as pd
cuda_metrics = pd.read_csv("cuda_metrics.csv")
required_columns = ["Matrix size", "CPU (ms)", "GPU kernel (ms)", "H2D+D2H (ms)",
                     "GPU end-to-end (ms)", "Speedup", "Max absolute error"]
assert list(cuda_metrics.columns) == required_columns
assert cuda_metrics["Matrix size"].tolist() == [256, 1024, 4096]
display(cuda_metrics.style.format({c: "{:.6g}" for c in required_columns[1:]}))

$ ./matrix_mul 256 1024 4096
GPU: Tesla T4
Block dimensions: 16 x 16 = 256 threads per block

N      CPU(ms)        Kernel(ms)     Transfer(ms)   EndToEnd(ms)     Speedup    MaxAbsErr     
  Running CPU reference for N=256 (3 repeats)...
256    11.1777        0.101101       0.668067       0.769168         14.5322    2.28882e-05   
  Running CPU reference for N=1024 (3 repeats)...
1024   305.807        4.92151        2.99691        7.91842          38.6197    9.15527e-05   
  Running CPU reference for N=4096 (3 repeats)...
4096   22593.2        244.861        45.4202        290.281          77.832     0.000366211   

Wrote cuda_metrics.csv



,Matrix size,CPU (ms),GPU kernel (ms),H2D+D2H (ms),GPU end-to-end (ms),Speedup,Max absolute error
0,256,11.1777,0.101101,0.668067,0.769168,14.5322,2.28882e-05
1,1024,305.807,4.92151,2.99691,7.91842,38.6197,9.15527e-05
2,4096,22593.2,244.861,45.4202,290.281,77.832,0.000366211


In [5]:
profiler_name, profile_command, extra_export = None, None, []

if shutil.which("ncu"):
    profiler_name = "Nsight Compute"
    profile_command = ["ncu", "--set", "basic", "./matrix_mul", "--profile", "1024"]
    extra_export = [
        ["ncu", "--set", "full", "--export", "hw1_ncu", "-f", "./matrix_mul", "--profile", "1024"],
    ]
elif shutil.which("nsys"):
    profiler_name = "Nsight Systems"
    profile_command = ["nsys", "profile", "--stats=true", "--force-overwrite=true",
                        "-o", "hw1_profile", "./matrix_mul", "--profile", "1024"]
elif shutil.which("nvprof"):
    profiler_name = "nvprof"
    profile_command = ["nvprof", "./matrix_mul", "--profile", "1024"]
else:
    profiler_name = "None available"

print("Profiler selected:", profiler_name)

if profile_command is not None:
    basic_run = run_command(profile_command, check=False)
    Path("profiler_output.txt").write_text(
        f"Profiler selected: {profiler_name}\n\n{basic_run.stdout}", encoding="utf-8"
    )
else:
    Path("profiler_output.txt").write_text(
        "No Nsight Systems, Nsight Compute, or nvprof executable was available in this Colab runtime.",
        encoding="utf-8",
    )

log("CUDA SECTION", f"Profiler selected: {profiler_name}")

Profiler selected: Nsight Compute
$ ncu --set basic ./matrix_mul --profile 1024
==PROF== Connected to process 7739 (/content/matrix_mul)
==PROF== Profiling "tiled_matrix_multiply" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "tiled_matrix_multiply" - 1: 0%....50%....100% - 9 passes
GPU: Tesla T4
Block dimensions: 16 x 16 = 256 threads per block
Profiling target: N=1024, grid=(64,64,1), block=(16,16,1), total threads=1048576
Profiling run N=1024 complete (GPU-only, no CPU reference computed). kernel=1668.14 ms, transfer=3.28554 ms
==PROF== Disconnected from process 7739
[7739] matrix_mul@127.0.0.1
  tiled_matrix_multiply(const float *, const float *, float *, int) (64, 64, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.99
  

In [6]:
if shutil.which("ncu"):
    run_command(["ncu", "--set", "full", "--export", "hw1_ncu", "-f",
                 "./matrix_mul", "--profile", "1024"])
    details_run = run_command(["ncu", "--import", "hw1_ncu.ncu-rep"], check=False)
    Path("ncu_details.txt").write_text(details_run.stdout, encoding="utf-8")
    assert Path("hw1_ncu.ncu-rep").exists(), "hw1_ncu.ncu-rep was not created"
    print("Wrote ncu_details.txt and hw1_ncu.ncu-rep")
else:
    Path("ncu_details.txt").write_text(
        "Nsight Compute (ncu) was not available in this Colab runtime; "
        "hw1_ncu.ncu-rep could not be produced.", encoding="utf-8"
    )
    print("ncu not available — see ncu_details.txt for the explanation.")

$ ncu --set full --export hw1_ncu -f ./matrix_mul --profile 1024
==PROF== Connected to process 7811 (/content/matrix_mul)
==PROF== Profiling "tiled_matrix_multiply" - 0: 0%....50%....100% - 31 passes
==PROF== Profiling "tiled_matrix_multiply" - 1: 0%....50%....100% - 31 passes
GPU: Tesla T4
Block dimensions: 16 x 16 = 256 threads per block
Profiling target: N=1024, grid=(64,64,1), block=(16,16,1), total threads=1048576
Profiling run N=1024 complete (GPU-only, no CPU reference computed). kernel=6532 ms, transfer=3.56211 ms
==PROF== Disconnected from process 7811
==PROF== Report: /content/hw1_ncu.ncu-rep

$ ncu --import hw1_ncu.ncu-rep
[7811] matrix_mul@127.0.0.1
  tiled_matrix_multiply(const float *, const float *, float *, int) (64, 64, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ----------

In [7]:
beneficial = cuda_metrics.loc[cuda_metrics["Speedup"] > 1.0, "Matrix size"]
if beneficial.empty:
    crossover = (
        "None of the tested matrix sizes (256, 1024, 4096) showed an end-to-end GPU benefit "
        "in this run. The crossover cannot be at size zero because a GPU operation carries "
        "fixed kernel-launch, synchronization, and transfer costs regardless of how little "
        "arithmetic it performs, so a larger workload would be needed before parallel "
        "computation outweighs those fixed costs."
    )
else:
    smallest = int(beneficial.min())
    row = cuda_metrics[cuda_metrics["Matrix size"] == smallest].iloc[0]
    crossover = (
        f"The smallest tested matrix size with an end-to-end GPU benefit was {smallest}, "
        f"with measured speedup {row['Speedup']:.2f}. At that size the parallel arithmetic "
        "was already large enough to outweigh transfer and launch overhead. The crossover is "
        "not at size zero because GPU work has fixed setup, launch, synchronization, and "
        "transfer costs that a near-zero-size problem cannot amortize."
    )
print(crossover)
log("CUDA SECTION", "Crossover analysis: " + crossover)

The smallest tested matrix size with an end-to-end GPU benefit was 256, with measured speedup 14.53. At that size the parallel arithmetic was already large enough to outweigh transfer and launch overhead. The crossover is not at size zero because GPU work has fixed setup, launch, synchronization, and transfer costs that a near-zero-size problem cannot amortize.


In [8]:
required_outputs = [Path("cuda_metrics.csv"), Path("cuda_run_output.txt"),
                    Path("profiler_output.txt"), Path("ncu_details.txt"), Path("RUN_LOG.txt")]
assert all(p.exists() and p.stat().st_size > 0 for p in required_outputs)
print("All CUDA artifacts present and non-empty.")
log("CUDA SECTION END", "All CUDA verification checks passed.")

All CUDA artifacts present and non-empty.
